In [1]:
# 神经网络 NN 参数
window_size = 16  #滑动窗口为4天，因为短期天气过程一般是为3-4天。
epochs = 200      #迭代次数
batch_size=32     #小规模数据一般建议选择32/64
val_ratio=0.2     #测试集为20%的数据

In [2]:
import time
import os, sys
import re #导入正则表达式模块，模式匹配、搜索和字符串操作

import numpy as np
import pandas as pd
from numpy import isnan
import random
import math
import xarray as xr
from netCDF4 import Dataset, num2date

import matplotlib
import matplotlib as mpl
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab #3D绘图
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm #色彩控制包
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import FormatStrFormatter
from matplotlib.font_manager import FontProperties
from matplotlib.dates import AutoDateLocator,DayLocator,MonthLocator,HourLocator, DateFormatter, drange

import seaborn as sns
import cmaps #conda install -c conda-forge cmaps
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

from joblib import Parallel, delayed
import itertools
import ipywidgets as widgets
from tqdm import tqdm  #进度条工具库，常用于显示循环的进度条
from subprocess import call
import glob
import pathlib 
from pathlib import Path
from copy import deepcopy

import datetime
from datetime import timedelta
from datetime import date
from datetime import datetime 

import torch
import torch.nn as nn
import torch.nn.functional
import torch.optim as optim
from torch.utils.data import Dataset as TorchDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import TimeSeriesSplit
from sklearn.mixture import GaussianMixture
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN

from scipy.stats import pearsonr
from scipy.stats import linregress
from scipy.optimize import curve_fit
from scipy import stats

import statsmodels.api as sm#用于统计建模、统计测试和数据探索可视化
from causal_ccm.causal_ccm import ccm #交叉收敛映射算法实现因果推断

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

matplotlib.use('Agg')#一切绘图均只输出在文件夹中
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['SimHei']
########################################################################
#用于将文件名中的“非法”字符替换成下划线 _，然后去掉开始和结尾的空白.
def safe_filename(s: str) -> str:#-> str为类型注解（return type annotation），不会执行。
	s = str(s)
	return "".join(c if (c.isalnum() or c in (' ', '-', '_')) else '_' for c in s).strip()

#提取某个完整文件路径的文件名
def get_filename(filename):
  (filepath,tempfilename) = os.path.split(filename);
  (shotname,extension) = os.path.splitext(tempfilename);
  #return filepath, shotname, extension
  return shotname
########################################################################

In [3]:
vas_Chem_sel=['O3_ML', 'O3_US', 'O3_LS', 'O3_TR','CH4_ML', 'CH4_US', 'CH4_LS', 'CH4_TR',#甲烷和臭氧是被大量观测的常规气体
              'CO_LT', 'CO_ML', 'CO_US', 'CO_LS', 'CO_TR',#只有NO和CO在低热层LT有明显浓度
              'NO_LT','NO_ML','NO_US', 'NO_LS', 'NO_TR',  #只有NO和CO在低热层LT有明显浓度
              'HO2NO2_ML', 'HO2NO2_US', 'HO2NO2_LS','HO2NO2_TR','HNO3_ML', 'HNO3_US','HNO3_LS', 'HNO3_TR',#含氮活性物种能到中间层
              'NO2_ML', 'NO2_US', 'NO2_LS', 'NO2_TR','N2O_ML', 'N2O_US', 'N2O_LS', 'N2O_TR',#含氮活性物种能到中间层
              'SO2_US','SO2_LS', 'SO2_TR','DMS_US','DMS_LS', 'DMS_TR'] #地面排放的硫化物气体到不了中间层

#选取第31层至70层，20公里高度以下，一般的航空和探空资料能覆盖，所以同化较为准确，且与对流层天气关系密切。
vas_Met=['RH_31', 'RH_32', 'RH_33', 'RH_34', 'RH_35', 'RH_36', 'RH_37', 'RH_38', 'RH_39', 'RH_40', 
         'RH_41', 'RH_42', 'RH_43', 'RH_44', 'RH_45', 'RH_46', 'RH_47', 'RH_48', 'RH_49', 'RH_50', 
         'RH_51', 'RH_52', 'RH_53', 'RH_54', 'RH_55', 'RH_56', 'RH_57', 'RH_58', 'RH_59', 'RH_60', 
         'RH_61', 'RH_62', 'RH_63', 'RH_64', 'RH_65', 'RH_66', 'RH_67', 'RH_68', 'RH_69', 'RH_70',
         'T_31', 'T_32', 'T_33', 'T_34', 'T_35', 'T_36', 'T_37', 'T_38', 'T_39', 'T_40', 
         'T_41', 'T_42', 'T_43', 'T_44', 'T_45', 'T_46', 'T_47', 'T_48', 'T_49', 'T_50', 
         'T_51', 'T_52', 'T_53', 'T_54', 'T_55', 'T_56', 'T_57', 'T_58', 'T_59', 'T_60', 
         'T_61', 'T_62', 'T_63', 'T_64', 'T_65', 'T_66', 'T_67', 'T_68', 'T_69', 'T_70']

vas_tianwen=['Moon_Phase','Moon_Light','Moon_Earth','Sun_Earth','Venus_Earth',
			 'Mercury_Earth','Mars_Earth','Jupiter_Earth','Saturn_Earth',
			 'Lon_Earth','Lon_Moon','Lon_Mercury','Lon_Venus','Lon_Mars',
             'Lon_Jupiter','Lon_Saturn','Mercury_Mars','Mercury_Venus','Mars_Venus']#金水火三星为近地行星，距离较近，单独算距离。

vas_ganzhi=['年干配数', '年支天数', '年支地数', '月干配数', '月支天数', '月支地数', 
			'日干配数', '日支天数', '日支地数']

vas_meihua=['本卦编码_0_encoded', '本卦编码_6_encoded', '本卦编码_12_encoded', '本卦编码_18_encoded', 
			'变卦编码_0_encoded', '变卦编码_6_encoded', '变卦编码_12_encoded', '变卦编码_18_encoded']

vas_obs=['Prec'] 

vas_X=vas_tianwen+vas_Chem_sel+vas_obs+vas_Met+vas_ganzhi+vas_meihua
print(len(vas_X))
vas_Y='tp'

xlsx_path_28="D://Data_and_Code//New_Data//Merged_WACCM_MERRA_ISD_SD//Sites_6H_28_sites//"#时间分辨率为每6小时

nc_ERA5_TP = "D:\Data_and_Code\ERA5_TP"  #总降水量的分布
nc_GRIDSAT="E:\GRIDSAT_2020_2024_Summer" #亮温的分布
#plot_output="D://Data_and_Code//New_Data//Merged_WACCM_MERRA_ISD_SD//Plots_2025-08-31//"
plot_output="D://Data_and_Code//New_Data//Merged_WACCM_MERRA_ISD_SD//Plots_Output//"

Site_11_C_name=['天文','北京', '南京', '广州', '上海', '武汉', '郑州', '昆明', '西安', '成都', '青岛']
Site_28_C_name=['天文','北京', '南京', '广州', '上海', '武汉', '郑州', '昆明', '西安', '成都', '青岛',
               '哈尔滨','长春','沈阳','太原','石家庄','呼和浩特','兰州','银川','乌鲁木齐',
               '重庆','合肥','济南','杭州','贵阳','长沙','南宁','福州']

157


In [4]:
station_meta = {
    '天文':    (30, 115, 1000000000),     #假设有一个太阳系内的观察站点
    
    '北京':    (39.48, 116.28, 32.5),     #54511
    '南京':    (31.56, 118.54, 36.4),     #58238
    '上海':    (31.24, 121.27, 6.7),      #58362
    '广州':    (23.13, 113.29, 71.5),     #59287
    '成都':    (30.45, 103.52, 548.9),    #56187
    '武汉':    (30.36, 114.03, 24.4),     #57494
    '西安':    (34.26, 108.58, 411.0),    #57131
    '昆明':    (25.04, 102.39, 1889.1),   #56778
    '郑州':    (34.43, 113.39, 111.6),    #57083
    '青岛':    (36.04, 120.20, 75.3),     #54857
    '哈尔滨':  (45.56, 126.34, 117.7),    #50953
    '长春':    (43.54, 125.13, 237.5),    #54161
    '沈阳':    (41.44, 123.31, 49.5),     #54342
    '太原':    (37.37, 112.35, 777.3),    #53772
    '石家庄':  (38.02, 114.50, 54.6),     #53698
    '呼和浩特':(40.51, 111.34, 1154.4),   #53463
    '兰州':    (35.52, 104.09, 1875.6),   #52983
    '银川':    (38.28, 106.12, 1111.6),   #53614
    '乌鲁木齐':(43.78, 87.65, 1928.5),    #51463
    '重庆':    (29.35, 106.28, 259.6),    #57516
    '合肥':    (31.47, 117.18, 28.2),     #58321
    '济南':    (36.36, 117.01, 171.2),    #54823
    '杭州':    (30.14, 120.10, 42.6),     #58457
    '贵阳':    (26.35, 106.44, 1224.9),   #57816
    '长沙':    (28.13, 112.55, 69.2),     #57687
    '南宁':    (22.38, 108.13, 122.6),    #59431
    '福州':    (26.05, 119.17, 84.8),     #58847
}

# LAT_MIN, LAT_MAX = 0, 60
# LON_MIN, LON_MAX = 70, 160
# HEIGHT_MIN, HEIGHT_MAX=2, 6000

In [5]:
# ==============================================================================
# === 第一部分：读入数据 & Step 1: 对数变换 ===
# ==============================================================================

# --- 读入X: 站点数据 ---
print("正在读取站点数据...")

#使用27+1个站点的数据
#file_paths = sorted(glob.glob(os.path.join(xlsx_path_28, "*.xlsx"))) 

#使用10+1个站点的数据
all_files = sorted(glob.glob(os.path.join(xlsx_path_28, "*.xlsx")))
file_paths = [
    f for f in all_files 
    if any(site in os.path.basename(f) for site in Site_11_C_name)
]

print(len(file_paths))
#print(file_paths)

dfs = []
for fp in file_paths:
    df_a = pd.read_excel(fp, parse_dates=['time'])
    df_a = df_a.set_index('time')

    valid_cols = [col for col in vas_X if col in df_a.columns]
    df = df_a[valid_cols].copy()
    
    if 'Prec' in df.columns:
        df.loc[:, 'Prec'] = df['Prec'].fillna(0)
        
    station_name = get_filename(fp)
    lat, lon, height = station_meta[station_name]
    station_prefix = get_filename(fp) + '_'
    df.columns = [station_prefix + str(col) for col in df.columns]
    print(df.shape)
    dfs.append(df)

df_merged = pd.concat(dfs, axis=1)
print(df_merged.shape)
df_merged = df_merged.sort_index().dropna()
print(df_merged.shape)

# --- 读入Y: NetCDF数据 ---
print("正在读取 NetCDF 数据...")
agg = 6
varname = 'tp'
time_var = 'valid_time'
nc_files = sorted(glob.glob(os.path.join(nc_ERA5_TP, "era5.mslp.*.nc")))

all_y = []
all_times_utc = []
lat_vals = None
lon_vals = None
time_unit = None
time_calendar = 'standard'

for f in tqdm(nc_files, desc='Reading nc files'):
    ds = Dataset(f)
    y = ds.variables[varname][:] 
    time_vals = ds.variables[time_var][:]
    
    if time_unit is None:
        time_unit = getattr(ds.variables[time_var], 'units', None)
    if hasattr(ds.variables[time_var], 'calendar'):
        time_calendar = getattr(ds.variables[time_var], 'calendar')
        
    times = num2date(time_vals, time_unit, time_calendar)
    all_y.append(y)
    all_times_utc.extend(times)
    
    if lat_vals is None:
        for nv in ('latitude', 'lat'):
            if nv in ds.variables:
                lat_vals = ds.variables[nv][:]
                break
    if lon_vals is None:
        for nv in ('longitude', 'lon'):
            if nv in ds.variables:
                lon_vals = ds.variables[nv][:]
                break
    ds.close()

y_all = np.concatenate(all_y, axis=0)
times_utc = np.array(all_times_utc)

# 聚合
n_hours = y_all.shape[0] // agg
y_agg = np.stack([y_all[i*agg:(i+1)*agg].sum(axis=0) for i in range(n_hours)], axis=0)

# 时间处理换算成和站点数据一样的北京时间
times_bj = np.array([dt + timedelta(hours=8) for dt in times_utc])
times_nc_agg = times_bj.reshape(-1, agg)[:, 0]
cftime_dt = [datetime(t.year, t.month, t.day, t.hour, t.minute, t.second) for t in times_nc_agg]

# 匹配索引
df_merged.index = pd.to_datetime(df_merged.index)
time_to_idx = {dt: idx for idx, dt in enumerate(cftime_dt)}
idx_list = [time_to_idx.get(dt, None) for dt in df_merged.index]
idx_arr = np.array([idx for idx in idx_list if idx is not None])

matched_times = [cftime_dt[i] for i in idx_arr]

# 单位转换与对数变换
y_agg_mm = y_agg * 1000 

# print("正在应用 Log(x+1) 变换...")
y_agg_log = np.log1p(y_agg_mm) # <---  y_agg_log = ln(1 + y_agg_mm)，ln 表示自然对数（底数 e）。
filtered_data = y_agg_log[idx_arr, :, :]

# 构造 xarray (Log 空间)，这一步只是备用，神经网络训练并未用到。
F_da = xr.DataArray(filtered_data,
                    coords=[pd.to_datetime(matched_times), lat_vals, lon_vals],
                    dims=['time', 'latitude', 'longitude'])

print(f"数据准备完成. 形状: {filtered_data.shape}. ")

正在读取站点数据...
11
(7670, 121)
(7670, 121)
(7670, 121)
(7670, 36)
(7670, 121)
(7670, 121)
(7670, 121)
(7670, 121)
(7670, 121)
(7670, 121)
(7670, 121)
(7670, 1246)
(2194, 1246)
正在读取 NetCDF 数据...


Reading nc files: 100%|█████████████████████████████████████████████████████████████████████████| 1918/1918 [02:02<00:00, 15.64it/s]


数据准备完成. 形状: (2194, 241, 361). 


In [6]:
# ==============================================================================
# === 神经网络 (CNN + L1 Loss) ===
# ==============================================================================

class Seq2GridDataset(TorchDataset):
    def __init__(self, df_x, y_grids, window_size):
        self.X = df_x.astype(np.float32)
        self.Y = y_grids.astype(np.float32)
        self.window = window_size
        self.feat_dim = self.X.shape[1]
        self.spatial_shape = self.Y.shape[1:]
        self.indices = list(range(self.window - 1, len(self.X))) 

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):       
        end = self.indices[idx]
        start = end - self.window + 1
        x_seq = self.X[start:end+1]
        y_grid = self.Y[end]
        return torch.tensor(x_seq), torch.tensor(y_grid)

# --- CNN Decoder 架构 ---

# === torch.nn.functional.interpolate ===

class Seq2GridNet_CNN(nn.Module):
    #LSTM隐藏单元，基线设定从 64、128、256 这样的中等大小开始。小数据、简单任务，128 以下往往足够，128 常作为稳健的起点。
    def __init__(self, input_dim, window, lstm_hidden=128, target_h=241, target_w=361):
        super().__init__()
        self.target_h = target_h
        self.target_w = target_w
        
        # LSTM: 提取时间序列特征
        self.lstm = nn.LSTM(input_dim, lstm_hidden, num_layers=2, batch_first=True, dropout=0.2)
        
        # 初始小图尺寸 (15x22 -> 约 1/16 的 240x352)
        self.init_h, self.init_w = 15, 22 
        self.channels = 64
        self.fc = nn.Linear(lstm_hidden, self.channels * self.init_h * self.init_w)
        
        # CNN Decoder: 逐步上采样恢复空间结构
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1), # ~30x44
            nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # ~60x88
            nn.BatchNorm2d(16), nn.ReLU(True),
            nn.ConvTranspose2d(16, 8, kernel_size=3, stride=2, padding=1, output_padding=1), # ~120x176
            nn.BatchNorm2d(8), nn.ReLU(True),
            nn.ConvTranspose2d(8, 8, kernel_size=3, stride=2, padding=1, output_padding=1), # ~240x352
            nn.BatchNorm2d(8), nn.ReLU(True),
            nn.Conv2d(8, 1, kernel_size=3, padding=1) # Final mapping
        )

    def forward(self, x):
        # x: (B, window, feat)
        lstm_out, _ = self.lstm(x) 
        x = lstm_out[:, -1, :] 
        
        x = self.fc(x)
        x = x.view(-1, self.channels, self.init_h, self.init_w)
        x = self.decoder(x) 
        
        # 使用 torch.nn.functional 全名
        x = torch.nn.functional.interpolate(x, size=(self.target_h, self.target_w), mode='bilinear', align_corners=False)
        
        return x.squeeze(1)
        
def train_and_evaluate_cnn(window=16, val_ratio=0.2, batch_size=32, epochs=50, device='cpu'):
    print('\n=== 开始训练神经网络 (CNN + L1Loss) ===')
    
    # 1. 数据准备
    scaler = StandardScaler()# Z-score标准正态化，z = (x - μ) / σ，μ 是该特征的样本均值，σ 是标准差。
    df_X = scaler.fit_transform(df_merged)
    # y_grids 已经是 Log 变换后的数据
    y_grids = filtered_data 
    
    ds = Seq2GridDataset(df_X, y_grids, window)
    n_val = int(len(ds) * val_ratio)
    n_train = len(ds) - n_val
    train_set, val_set = torch.utils.data.random_split(ds, [n_train, n_val])
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    
    # 2. 模型构建
    feat_dim = ds.feat_dim
    H, W = ds.spatial_shape
    model = Seq2GridNet_CNN(feat_dim, window, lstm_hidden=128, target_h=H, target_w=W).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    # <--- 优化: 使用 L1 Loss，计算输出 y_pred 与真实值 y_true 之间的绝对误差的均值
    # 对异常值的不敏感性相对低于 L2 损失（MSE），对离群点的鲁棒性较好。
    criterion = nn.L1Loss() 
    
    # 3. 训练
    model.train()
    train_losses = []
    for epoch in range(epochs):
        total_loss = 0
        for xseq, ygrid in train_loader:
            xseq, ygrid = xseq.to(device), ygrid.to(device)
            optimizer.zero_grad()
            pred = model(xseq)
            loss = criterion(pred, ygrid)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xseq.size(0)
        
        avg_loss = total_loss / n_train
        train_losses.append(avg_loss)
        
        #print(f'Epoch {epoch+1}/{epochs} Train MAE: {avg_loss:.5f}')#每一次训练都要打印
        # === 修改处：每 10 个 epoch 打印一次 ===
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1}/{epochs} Train MAE: {avg_loss:.5f}')
            
    # 4. 验证 & 逆变换评估
    model.eval()#把模型切换到评估/推理模式（evaluation mode）
    '''
    在训练模式（model.train()）下，Dropout 会按设定的概率随机“丢弃”一部分神经元，以防止过拟合。
    在评估模式下，Dropout 不会丢弃神经元，而是以全部神经元工作，并且通常会按训练时的缩放规则进行等效放大。
    训练时，BatchNorm 使用当前小批量数据计算均值与方差并更新其移动平均参数。
    评估时，BatchNorm 使用在训练过程中学习到的全局均值和方差（即移动均值/方差），以获得稳定的推理结果。
    '''
    
    preds_log = []
    trues_log = []
    with torch.no_grad():#禁用梯度计算，减少内存占用和加速推理
        for xseq, ygrid in tqdm(val_loader, desc='Eval'):
            xseq = xseq.to(device)
            pred = model(xseq)
            preds_log.append(pred.cpu().numpy())
            trues_log.append(ygrid.numpy()) # cpu not needed for tensor created from numpy
            
    preds_log = np.concatenate(preds_log, axis=0)
    trues_log = np.concatenate(trues_log, axis=0)
    
    # <--- 逆变换回 mm 单位进行评估
    # print("正在将神经网络结果逆变换回 mm 单位...")
    preds_mm = np.expm1(preds_log)
    trues_mm = np.expm1(trues_log)

     # 避免出现负数雨量 (模型可能输出微小的负数)
    preds_mm = np.maximum(preds_mm, 0)
    
    # 计算全场均值用于求系统偏差
    p_mean = preds_mm.mean(axis=(1,2))
    t_mean = trues_mm.mean(axis=(1,2))
    #拟合线性关系 (Pred = slope * True + intercept),polyfit(x, y) 的 x 是真值，y 是预测值
    slope, intercept = np.polyfit(t_mean, p_mean, 1)
    #preds_mm = preds_mm*1.5 #Log变换后，根据詹森不等式，神经网络模型容易整体低估强降水，也可将预测值乘以系数1.5，但RMSE误差会增大。

    rmse = np.sqrt(np.mean((preds_mm - trues_mm) ** 2))
    mae = np.mean(np.abs(preds_mm - trues_mm))

    # 计算平均空间相关系数，注意这个空间相关是直接计算每一个网格点的真实值与模拟值两列相关性
    corrs = []
    for i in range(preds_mm.shape[0]):
        p = preds_mm[i].flatten()
        t = trues_mm[i].flatten()
        # 避免全0导致的 NaN
        if np.std(t) > 1e-5 and np.std(p) > 1e-5:
            corrs.append(np.corrcoef(p, t)[0, 1])
            
    spatial_corr = np.nanmean(corrs)
    print(f"Val RMSE: {rmse:.4f}  MAE: {mae:.4f}  平均空间相关: {spatial_corr:.4f}")
    
    #return model, train_losses, preds_mm, trues_mm
    return model, train_losses, preds_mm, trues_mm #, corrs, slope, intercept

In [7]:
# 运行训练
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, train_losses, preds_eval, trues_eval= train_and_evaluate_cnn(
                                              window=window_size, val_ratio=val_ratio, batch_size=batch_size, epochs=epochs, device=device)


=== 开始训练神经网络 (CNN + L1Loss) ===
Epoch 10/200 Train MAE: 0.30032
Epoch 20/200 Train MAE: 0.25500
Epoch 30/200 Train MAE: 0.23012
Epoch 40/200 Train MAE: 0.21509
Epoch 50/200 Train MAE: 0.20446
Epoch 60/200 Train MAE: 0.19677
Epoch 70/200 Train MAE: 0.19163
Epoch 80/200 Train MAE: 0.18676
Epoch 90/200 Train MAE: 0.18296
Epoch 100/200 Train MAE: 0.17991
Epoch 110/200 Train MAE: 0.17696
Epoch 120/200 Train MAE: 0.17429
Epoch 130/200 Train MAE: 0.17230
Epoch 140/200 Train MAE: 0.17060
Epoch 150/200 Train MAE: 0.16891
Epoch 160/200 Train MAE: 0.16735
Epoch 170/200 Train MAE: 0.16747
Epoch 180/200 Train MAE: 0.16437
Epoch 190/200 Train MAE: 0.16341
Epoch 200/200 Train MAE: 0.16205


Eval: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 14/14 [00:01<00:00,  8.49it/s]


Val RMSE: 2.9011  MAE: 1.0816  平均空间相关: 0.5665


注：训练耗时约60分钟

In [8]:
from skimage.metrics import structural_similarity as ssim

# === 1. FSS 计算函数 ===
def compute_fss(obs, pred, threshold, window_size):
    """
    计算分数技巧评分 (FSS)
    obs, pred: 2D numpy array (H, W)
    threshold: 降水阈值 (例如 10mm)
    window_size: 邻域窗口大小 (奇数，例如 5, 9, 15)
    """
    # 二值化
    obs_binary = (obs >= threshold).astype(np.float32)
    pred_binary = (pred >= threshold).astype(np.float32)
    
    # 使用均值滤波器计算邻域内的发生比例 (Fraction)
    # 利用 scipy 的 uniform_filter 或者简单的卷积
    from scipy.ndimage import uniform_filter
    
    obs_frac = uniform_filter(obs_binary, size=window_size, mode='constant', cval=0)
    pred_frac = uniform_filter(pred_binary, size=window_size, mode='constant', cval=0)
    
    # 计算 MSE (Mean Squared Error of Fractions)
    mse = np.nanmean((obs_frac - pred_frac)**2)
    
    # 计算参考 MSE (即最大的可能误差: 分子部分)
    # FSS 公式定义为: mean(obs_frac^2 + pred_frac^2)
    mse_ref = np.nanmean(obs_frac**2 + pred_frac**2)
    
    if mse_ref == 0:
        return 0 # 或者是 NaN，视情况而定
        
    fss = 1 - (mse / mse_ref)
    return fss

# === 2. SSIM 计算函数 ===
def compute_spatial_metrics(preds_mm, trues_mm):
    """
    批量计算多种空间评估指标
    """
    pccs = []
    ssims = []
    fss_scores = [] 
    
    threshold = 10.0 # 关注 10 mm 以上的降水
    window = 9       # 窗口大小
    
    # SSIM 需要指定数据范围 data_range，这里取全局最大值作为参考
    max_val = max(np.max(preds_mm), np.max(trues_mm))
    
    for i in range(preds_mm.shape[0]):
        p = preds_mm[i]
        t = trues_mm[i]
        
        # 1. PCC (你原来的方法)
        if np.std(t) > 1e-5 and np.std(p) > 1e-5:
            pccs.append(np.corrcoef(p.flatten(), t.flatten())[0, 1])
        
        # 2. SSIM
        # win_size 必须小于图像边长，默认7或者更小
        try:
            s = ssim(t, p, data_range=max_val, win_size=7) 
            ssims.append(s)
        except ValueError:
            pass
            
        # 3. FSS
        f = compute_fss(t, p, threshold=threshold, window_size=window)
        fss_scores.append(f)
        
    return np.nanmean(pccs), np.nanmean(ssims), np.nanmean(fss_scores)

In [9]:
# === 调用 ===
mean_pcc, mean_ssim, mean_fss = compute_spatial_metrics(preds_eval, trues_eval)

print(f"评价指标:")
print(f"PCC (模式相关): {mean_pcc:.4f} (关注整体形态)")
print(f"SSIM (结构相似): {mean_ssim:.4f} (关注视觉感知)")
print(f"FSS (分数技巧): {mean_fss:.4f} (关注明显降水落区容错, 阈值=10 mm)")

评价指标:
PCC (模式相关): 0.5665 (关注整体形态)
SSIM (结构相似): 0.9085 (关注视觉感知)
FSS (分数技巧): 0.4443 (关注明显降水落区容错, 阈值=10 mm)


In [10]:
# --- 绘图：观测 vs 模拟 (离散色标) ---
def plot_example_maps(obs_da, recon_da, times_to_plot, outpath,
                      figsize_per_row=(10, 4),
                      coast_resolution='50m',
                      **kwargs): # 接收多余参数防止报错
    
    n = len(times_to_plot)
    if n == 0: return

    # --- A. 定义降水专用离散色标 (Custom Precipitation Colormap) ---
    # 颜色列表 (11种颜色，对应参考图风格)
    colors = ["cyan", "darkcyan", "green", "lime", 
              "yellow", "gold", "goldenrod", 
              "darkred", "brown", "red", "darkviolet"]#共11个配色
    
    # 创建 Colormap 对象
    cmap = mpl.colors.ListedColormap(colors)
    cmap.set_over("indigo")  # 超过最大值显示靛蓝
    cmap.set_under("white")  # 低于最小值显示白色

    # 定义分级阈值 (Bounds)
    #bounds = [0.1, 0.5, 1, 2, 3, 4, 5, 6, 8, 10, 20, 40]#共12个等级11个配色
    bounds = [2, 5, 10, 15, 20, 25, 30, 40, 50, 60, 70, 80]
    
    # 检查颜色和区间数量是否匹配 (颜色数应 = 边界数 - 1)
    if len(bounds) - 1 != len(colors):
        print(f"警告：颜色数量({len(colors)})与区间数量({len(bounds)-1})不匹配，将自动调整。")
        # 兜底逻辑：截取或插值 (这里简单截取以防报错)
        bounds = bounds[:len(colors)+1]

    # 创建 Norm (将数值映射到离散颜色索引)
    norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

    # --- B. 开始绘图 ---
    fig, axes = plt.subplots(nrows=n, ncols=2,
                             figsize=(figsize_per_row[0], figsize_per_row[1] * n),
                             subplot_kw={'projection': ccrs.PlateCarree()})
    
    if n == 1: axes = np.array([axes])
    
    # 布局调整：预留右侧空白给 Colorbar
    fig.subplots_adjust(right=0.88, hspace=0.3, wspace=0.2)

    for i, t in enumerate(times_to_plot):
        ax_obs = axes[i, 0]
        ax_rec = axes[i, 1]

        # 数据提取
        try:
            d_obs = obs_da.sel(time=t).squeeze().values
            d_rec = recon_da.sel(time=t).squeeze().values
            lon = obs_da.longitude.values
            lat = obs_da.latitude.values
        except Exception as e:
            print(f"Error {t}: {e}")
            continue

        # 绘图参数 (使用 cmap 和 norm)
        plot_kwargs = {
            'transform': ccrs.PlateCarree(),
            'cmap': cmap,
            'norm': norm, # <--- 关键：使用 BoundaryNorm
            'shading': 'auto'
        }

        # 绘制 Pcolormesh
        im1 = ax_obs.pcolormesh(lon, lat, d_obs, **plot_kwargs)
        im2 = ax_rec.pcolormesh(lon, lat, d_rec, **plot_kwargs)
        
        # 标题
        t_str = pd.to_datetime(t).strftime("%Y-%m-%d %H:%M")
        ax_obs.set_title(f'Precipitation (Obs): {t_str}', fontsize=12)#\nMax: {np.max(d_obs):.1f} mm
        ax_rec.set_title(f'Precipitation (Sim): {t_str}', fontsize=12)#\nMax: {np.max(d_rec):.1f} mm

        # 地图要素
        for ax in [ax_obs, ax_rec]:
            ax.coastlines(resolution=coast_resolution, linewidth=0.8, color='black')
            #ax.add_feature(cfeature.BORDERS, linewidth=0.5, color='gray')#不绘制有争议的国界线
            
            #绘制经纬度刻度线
            gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                              linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            
            # 1. 关闭顶部和右侧的标签 (符合学术绘图习惯)
            gl.top_labels = False
            gl.right_labels = False
            
            # 2. 设置标签格式 (显示 °E, °N)
            # gl.xformatter = LONGITUDE_FORMATTER
            # gl.yformatter = LATITUDE_FORMATTER
            
            # 3. 设置字体大小和颜色
            gl.xlabel_style = {'size': 10, 'color': 'black'}
            gl.ylabel_style = {'size': 10, 'color': 'black'}

        # --- C. Colorbar (保持位置不变) ---
        pos = ax_rec.get_position()
        cax_x = pos.x1 + 0.015  # 保持满意的距离
        cax_y = pos.y0
        cax_w = 0.015
        cax_h = pos.height
        
        cax = fig.add_axes([cax_x, cax_y, cax_w, cax_h])
        
        # extend='both' 也就是让 <2 mm 的显示白色(under)，>80 mm的显示靛蓝(over)
        # spacing='uniform' 让色条上的每一格高度相等，方便阅读（即使数值跨度是非均匀的）
        cbar = fig.colorbar(im2, cax=cax, orientation='vertical', 
                            extend='both', spacing='uniform', ticks=bounds)
        
        cbar.set_label('Precipitation (mm)', fontsize=10)
        cbar.ax.tick_params(labelsize=8)

    plt.savefig(outpath, dpi=300)
    plt.close()
    print(f"对比图(离散色标版)已保存至: {outpath}")

# --- 2. 差值图---
def plot_difference_maps(obs_da, recon_da, times_to_plot, outpath, figsize_per_row=(7, 5), coast_resolution='50m'):
    diff_da = recon_da - obs_da
    vals_all = []
    for t in times_to_plot:
        try: vals_all.append(diff_da.sel(time=t).values)
        except: pass
    
    if not vals_all: return
    vals_abs = np.abs(np.concatenate(vals_all))
    
    # 差值图使用 99% 分位数
    limit = np.nanpercentile(vals_abs, 99) 
    
    if limit < 0.1: limit = np.nanmax(vals_abs)
    if limit < 1e-3: limit = 1.0
    
    print(f"差值图 Colorbar Limit: ±{limit:.2f} mm")

    n = len(times_to_plot)
    fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(figsize_per_row[0], figsize_per_row[1]*n),
                             subplot_kw={'projection': ccrs.PlateCarree()})
    if n==1: axes=[axes]
    
    fig.subplots_adjust(right=0.90) # 留出空间

    for i, t in enumerate(times_to_plot):
        ax = axes[i]
        try:
            data = diff_da.sel(time=t).values
            lon = diff_da.longitude.values
            lat = diff_da.latitude.values
            
            im = ax.pcolormesh(lon, lat, data, transform=ccrs.PlateCarree(),
                               cmap='RdBu', vmin=-limit, vmax=limit, shading='auto')
            ax.coastlines(resolution=coast_resolution)
            #ax.add_feature(cfeature.BORDERS, linewidth=0.5, color='gray')
            
            #绘制经纬度刻度线
            gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                              linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            
            # 1. 关闭顶部和右侧的标签 (符合学术绘图习惯)
            gl.top_labels = False
            gl.right_labels = False
            
            # 2. 设置标签格式 (显示 °E, °N)
            # gl.xformatter = LONGITUDE_FORMATTER
            # gl.yformatter = LATITUDE_FORMATTER
            
            # 3. 设置字体大小和颜色
            gl.xlabel_style = {'size': 10, 'color': 'black'}
            gl.ylabel_style = {'size': 10, 'color': 'black'}
            
            # 4. (可选) 自定义刻度间隔
            # 如果自动生成的刻度太密或太稀，取消下面两行的注释并修改数值
            # import matplotlib.ticker as mticker
            # gl.xlocator = mticker.MultipleLocator(10) # 每隔10经度画线
            # gl.ylocator = mticker.MultipleLocator(5)  # 每隔5纬度画线
            # =======================================================
            
            t_str = pd.to_datetime(t).strftime("%Y-%m-%d %H:%M")
            
            max_err = np.max(np.abs(data))
            ax.set_title(f'Diff (Sim-Obs): {t_str} ', fontsize=12)#| Max Err: {max_err:.1f} mm
            
            cbar = plt.colorbar(im, ax=ax, shrink=0.9, pad=0.03, extend='both')
            cbar.set_label('Precipitation (mm)', fontsize=10)
            cbar.ax.tick_params(labelsize=8)
            
        except Exception as e: print(e)
    plt.savefig(outpath, dpi=300)
    plt.close()
    print(f"差值图已保存至: {outpath}")

# --- 3. 主执行函数 ---
def visualize_nn_predictions_final(model, target_times, df_predictors, y_truth_log, 
                                  time_list, lat_coords, lon_coords, 
                                  window_size, out_dir): 
    print("\n=== 生成最终预测图 (含逆变换) ===")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_predictors)
    
    time_index_map = {pd.to_datetime(t): i for i, t in enumerate(time_list)}
    
    obs_list = []
    recon_list = []
    valid_times = []
    
    model.eval()
    with torch.no_grad():
        for t_str in target_times:
            t_query = pd.to_datetime(t_str)
            if t_query not in time_index_map: continue
            idx = time_index_map[t_query]
            if idx < window_size - 1: continue
            
            x_seq = X_scaled[idx-window_size+1 : idx+1]
            x_tensor = torch.tensor(x_seq, dtype=torch.float32).unsqueeze(0).to(device)
            
            pred_log = model(x_tensor).cpu().numpy().squeeze(0)
            true_log = y_truth_log[idx]
            
            pred_mm = np.expm1(pred_log)
            true_mm = np.expm1(true_log)
            pred_mm = np.maximum(pred_mm, 0)
            #pred_mm = 1.5*pred_mm #系统性低估，乘以修正系数。
        
            obs_list.append(true_mm)
            recon_list.append(pred_mm)
            valid_times.append(t_query)
            
    if not valid_times: 
        print("无有效数据。")
        return

    da_obs = xr.DataArray(np.stack(obs_list), coords=[valid_times, lat_coords, lon_coords], 
                          dims=['time', 'latitude', 'longitude'], name='Obs_mm')
    da_recon = xr.DataArray(np.stack(recon_list), coords=[valid_times, lat_coords, lon_coords], 
                            dims=['time', 'latitude', 'longitude'], name='Pred_mm')
    
    print("正在绘制对比图 (CNN_Comparison.png)...")
    
    # 这里使用 pct=(0, 100) 来确保包含最大值
    plot_example_maps(da_obs, da_recon, valid_times, 
                      os.path.join(out_dir, 'CNN_Comparison.png'),
                      share_obs_recon=True, 
                      use_percentile=True, 
                      pct=(0, 100)) # 
        
    print("正在绘制差值图 (CNN_Diff.png)...")
    plot_difference_maps(da_obs, da_recon, valid_times, 
                         os.path.join(out_dir, 'CNN_Diff.png'))
    print("绘图完成.")

In [11]:
interested_times = ['2021-07-20 14:00', '2023-07-31 14:00', '2025-07-28 14:00'] #华北三次极端暴雨过程
visualize_nn_predictions_final(model, interested_times, df_merged, filtered_data, 
                               matched_times, lat_vals, lon_vals, window_size, plot_output)


=== 生成最终预测图 (含逆变换) ===
正在绘制对比图 (CNN_Comparison.png)...
对比图(离散色标版)已保存至: D://Data_and_Code//New_Data//Merged_WACCM_MERRA_ISD_SD//Plots_Output//CNN_Comparison.png
正在绘制差值图 (CNN_Diff.png)...
差值图 Colorbar Limit: ±10.84 mm
差值图已保存至: D://Data_and_Code//New_Data//Merged_WACCM_MERRA_ISD_SD//Plots_Output//CNN_Diff.png
绘图完成.


以上代码神经网络预测的是当前时刻的数据，评估的是测试集，这是 “同步重构” (Reconstruction / Nowcasting)。

以下是全部数据评估

In [12]:
def get_all_predictions(model, dataset, batch_size, device):
    # 1. 创建一个不打乱的、加载全量数据的 DataLoader
    all_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    model.eval()
    all_preds = []
    all_trues = []
    
    print("正在对全量数据进行推理...")
    with torch.no_grad():
        for xseq, ygrid in tqdm(all_loader, desc='Full Inference'):
            xseq = xseq.to(device)
            pred = model(xseq)
            all_preds.append(pred.cpu().numpy())
            all_trues.append(ygrid.numpy())
            
    # 拼接
    all_preds = np.concatenate(all_preds, axis=0)
    all_trues = np.concatenate(all_trues, axis=0)
    
    # 逆变换
    return np.expm1(all_preds), np.expm1(all_trues)

In [13]:
# ==============================================================================
# === 在全局范围重新构建数据集 ===
# ==============================================================================

# 1. 重新准备输入数据 (保持与训练时一致的预处理)
# 注意：这里必须对 df_merged 做标准化，因为模型是在标准化数据上训练的
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_X_scaled = scaler.fit_transform(df_merged)

# y_grids 就是之前的 filtered_data (已经在 Log 空间)
y_grids_log = filtered_data 

# 2. 实例化 PyTorch 数据集 (使用一个新名字，避免混淆)
# 确保 window_size 和你训练时用的一样
full_dataset = Seq2GridDataset(df_X_scaled, y_grids_log, window_size)

print(f"全量数据集构建完成，样本数: {len(full_dataset)}")

# 3. 现在调用之前训练好的预测模型，传入 dataset 对象
all_preds_mm, all_trues_mm = get_all_predictions(model, full_dataset, batch_size, device)
all_preds_mm=all_preds_mm*1.2

# 4. 检查结果
print(f"全量预测完成。")
print(f"预测数据形状: {all_preds_mm.shape}")
print(f"真值数据形状: {all_trues_mm.shape}")

# 5. 再次计算一下全量数据的评估指标
full_rmse = np.sqrt(np.mean((all_preds_mm - all_trues_mm) ** 2))
full_mae = np.mean(np.abs(all_preds_mm - all_trues_mm))
print(f"全量数据评估 -> RMSE: {full_rmse:.4f}, MAE: {full_mae:.4f}")

# ==============================================================================
# === 计算全量数据的平均空间相关性 (Spatial PCC) ===
# ==============================================================================
all_corrs = []

# 遍历每一个时间步
for i in range(all_preds_mm.shape[0]):
    # 1. 将当前时刻的 (H, W) 二维图展平为一维向量
    p_flat = all_preds_mm[i].flatten()
    t_flat = all_trues_mm[i].flatten()
    
    # 2. 安全检查：只有当真值和预测值都有波动(标准差>0)时才计算相关
    # 如果全场都是0（无雨），相关系数无定义（分母为0），应跳过
    if np.std(t_flat) > 1e-5 and np.std(p_flat) > 1e-5:
        # np.corrcoef 返回相关系数矩阵 [[1, r], [r, 1]]，取 [0,1] 元素
        corr = np.corrcoef(p_flat, t_flat)[0, 1]
        all_corrs.append(corr)

# 3. 计算平均值 (使用 nanmean 忽略潜在的 NaN)
full_spatial_corr = np.nanmean(all_corrs)

print(f"全量数据评估 -> 平均空间相关系数 (PCC): {full_spatial_corr:.4f}")
print(f"有效计算帧数: {len(all_corrs)} / {all_preds_mm.shape[0]}")
# 69 代表的是 Batch 的数量（Number of Batches）。batch_size = 32。样本总数: 69 * 32 ≈ 2200 个样本左右

全量数据集构建完成，样本数: 2179
正在对全量数据进行推理...


Full Inference: 100%|███████████████████████████████████████████████████████████████████████████████| 69/69 [00:07<00:00,  8.70it/s]


全量预测完成。
预测数据形状: (2179, 241, 361)
真值数据形状: (2179, 241, 361)
全量数据评估 -> RMSE: 2.0839, MAE: 0.7179
全量数据评估 -> 平均空间相关系数 (PCC): 0.7984
有效计算帧数: 2179 / 2179


使用 “置换特征重要性” (Permutation Feature Importance) 方法来后天计算出重要性。这是一个训练后（Post-hoc）的分析步骤。不需要重新训练。

In [14]:
# ==============================================================================
# === 特征重要性分析 (Permutation Importance) ===
# ==============================================================================
def calculate_permutation_importance(model, dataset, feature_names, device, batch_size=32):
    """
    计算特征重要性 (分批计算，防止内存OOM)
    """
    model.eval()
    criterion = nn.L1Loss(reduction='sum') # 注意：用 sum 以便累加
    #用了 sum reduction，那么这个 Loss 是所有像素点误差的总和。
    #如果是全场像素的总误差和，那么平均每个像素只有约 0.18 的误差
    
    # 1. 计算基准误差 (Baseline Loss)，遍历 DataLoader 累加误差
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    base_loss_sum = 0
    n_samples = 0
    
    print("正在计算基准误差...")
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            base_loss_sum += loss.item()
            n_samples += x.size(0)
            
    base_mae = base_loss_sum / n_samples
    print(f"基准 MAE: {base_mae:.5f}")
    
    # 2. 逐个特征进行置换
    # 我们需要知道特征的总数。取一个样本看看形状即可。
    dummy_x, _ = dataset[0]
    n_features = dummy_x.shape[1] # (Window, Feats) -> Feats在第1维
    
    importances = []
    
    for i in range(n_features):
        # 对于每一个特征，我们需要重新遍历整个 DataLoader
        # 并在每个 Batch 内部进行打乱
        
        perm_loss_sum = 0
        
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                
                # === 核心操作：只在当前 Batch 内打乱特征 ===
                # 这种近似方法在 Batch Size 足够大(>=32)时是有效的
                # 相比于全量打乱，这种方法极省内存
                x_permuted = x.clone()
                idx = torch.randperm(x.size(0))
                x_permuted[:, :, i] = x_permuted[idx, :, i]
                
                pred_perm = model(x_permuted)
                loss = criterion(pred_perm, y)
                perm_loss_sum += loss.item()
        
        perm_mae = perm_loss_sum / n_samples
        diff = perm_mae - base_mae
        importances.append(diff)
        
        if (i+1) % 20 == 0:
            print(f"已分析特征 {i+1}/{n_features}...")

    return np.array(importances)

# # 1. 获取特征名称列表
feature_names = list(df_merged.columns) 

# # 2. 计算重要性 (使用全量数据集 full_dataset 或 验证集 val_subset)
imp_scores = calculate_permutation_importance(model, full_dataset, feature_names, device)

正在计算基准误差...
基准 MAE: 16288.26680
已分析特征 20/1246...
已分析特征 40/1246...
已分析特征 60/1246...
已分析特征 80/1246...
已分析特征 100/1246...
已分析特征 120/1246...
已分析特征 140/1246...
已分析特征 160/1246...
已分析特征 180/1246...
已分析特征 200/1246...
已分析特征 220/1246...
已分析特征 240/1246...
已分析特征 260/1246...
已分析特征 280/1246...
已分析特征 300/1246...
已分析特征 320/1246...
已分析特征 340/1246...
已分析特征 360/1246...
已分析特征 380/1246...
已分析特征 400/1246...
已分析特征 420/1246...
已分析特征 440/1246...
已分析特征 460/1246...
已分析特征 480/1246...
已分析特征 500/1246...
已分析特征 520/1246...
已分析特征 540/1246...
已分析特征 560/1246...
已分析特征 580/1246...
已分析特征 600/1246...
已分析特征 620/1246...
已分析特征 640/1246...
已分析特征 660/1246...
已分析特征 680/1246...
已分析特征 700/1246...
已分析特征 720/1246...
已分析特征 740/1246...
已分析特征 760/1246...
已分析特征 780/1246...
已分析特征 800/1246...
已分析特征 820/1246...
已分析特征 840/1246...
已分析特征 860/1246...
已分析特征 880/1246...
已分析特征 900/1246...
已分析特征 920/1246...
已分析特征 940/1246...
已分析特征 960/1246...
已分析特征 980/1246...
已分析特征 1000/1246...
已分析特征 1020/1246...
已分析特征 1040/1246...
已分析特征 1060/1246...
已分析特征 1080/124

In [15]:
# ==============================================================================
# === 特征名解析与聚合 ===
# ==============================================================================

# 1. 构造一个 DataFrame 方便处理
df_imp = pd.DataFrame({
    'Full_Name': feature_names,
    'Score': imp_scores
})

def parse_feature_name(full_name):
    """
    智能解析 '站点_变量名' 格式
    假设：第一个下划线之前的是站点名，之后的所有内容都是变量名
    """
    # maxsplit=1: 只切一刀。
    # 例如 "54511_Temp_850hPa" -> ["54511", "Temp_850hPa"]
    parts = full_name.split('_', 1) 
    
    if len(parts) == 2:
        station = parts[0]
        variable = parts[1]
    else:
        # 容错处理：如果没有下划线
        station = "Unknown"
        variable = full_name
        
    return pd.Series([station, variable])

# 2. 应用解析逻辑
df_imp[['Station', 'Variable']] = df_imp['Full_Name'].apply(parse_feature_name)

# 3. 聚合计算
# (1) 站点重要性：按 Station 分组求和
station_importance = df_imp.groupby('Station')['Score'].sum().sort_values(ascending=False)

# (2) 变量重要性：按 Variable 分组求和
#    注意：这里会把不同站点的同一个变量加在一起，反映该物理变量的整体重要性
variable_importance = df_imp.groupby('Variable')['Score'].sum().sort_values(ascending=False)

# ==============================================================================
# === 绘图：Top-N 显示与清晰标签 ===
# ==============================================================================

# --- 图1: 站点重要性  ---
plt.figure(figsize=(8, 6), dpi=300)
# 使用 reset_index 方便 seaborn 绘图
data_station = station_importance.reset_index()

sns.barplot(data=data_station, x='Score', y='Station', hue='Station', palette='viridis')

plt.title("Station Importance (Total Impact)")
plt.xlabel("Cumulative Importance Score")
plt.ylabel("Station Name")
plt.tight_layout()
plt.savefig(os.path.join(plot_output, 'Importance_Station.png'))
print(f"站点数量: {len(data_station)}") # 应该正好是 10 个左右

# --- 图2: 变量重要性 (只画 Top 30) ---
plt.figure(figsize=(10, 8), dpi=300) # 画布调高一点

# 只取前 30 个最重要的变量，防止标签重叠
top_n = 30
data_var = variable_importance.head(top_n).reset_index()

sns.barplot(data=data_var, x='Score', y='Variable', hue='Variable',palette='magma') #hue=station_importance.index,legend=False

plt.title(f"Top {top_n} Variable Importance", fontsize=20)
plt.xlabel("Cumulative Importance Score", fontsize=18)
plt.ylabel("Variable Name", fontsize=18)
plt.tight_layout()
plt.savefig(os.path.join(plot_output, 'Importance_Variable_Top30.png'))

print("\n=== 分析结果摘要 ===")
print("最重要站点 Top3:")
print(station_importance.head(3))
print("\n最重要变量 Top5:")
print(variable_importance.head(10))

站点数量: 11

=== 分析结果摘要 ===
最重要站点 Top3:
Station
广州    46.386756
昆明    43.166081
北京    42.467309
Name: Score, dtype: float64

最重要变量 Top5:
Variable
DMS_LS       21.960332
DMS_US       15.526023
DMS_TR       14.747813
Prec         14.559144
SO2_LS        8.387312
RH_43         8.195786
RH_45         7.339519
NO_ML         7.016073
HO2NO2_LS     6.878983
HO2NO2_TR     6.715391
Name: Score, dtype: float64
